# Feature Engineering

Feature Engineering mainly consists of **four** types:
- ### Feature Transformation  
  > (Missing Value Imputation ,Handling Categorical Features, Outlier Detection, Feature Scaling)
- ### Feature Construction
- ### Feature Selection
- ### Feature Extraction

#### Today's Topic
- Feature Construction

In [1]:
import numpy as np 
import pandas as pd 
import seaborn as sns 

from sklearn.model_selection import cross_val_score 
from sklearn.linear_model import LogisticRegression 

In [2]:
df = pd.read_csv('train.csv',usecols=['Age','Pclass','SibSp','Parch','Survived'])
df.head()

,Survived,Pclass,Age,SibSp,Parch
0,0,3,22.0,1,0
1,1,1,38.0,1,0
2,1,3,26.0,0,0
3,1,1,35.0,1,0
4,0,3,35.0,0,0


In [3]:
# General Model Testing (Original Data set)

In [4]:
# Dropping all null values 
df.dropna(inplace=True)

In [5]:
x = df.iloc[:,1:]
y = df['Survived']

In [6]:
x.sample(5)

,Pclass,Age,SibSp,Parch
534,3,30.0,0,0
588,3,22.0,0,0
216,3,27.0,0,0
111,3,14.5,1,0
467,1,56.0,0,0


In [7]:
np.mean(cross_val_score(LogisticRegression(),x,y,scoring='accuracy',cv=20))

np.float64(0.6933333333333332)

# Note

#### Why we have not done Train-test-split before Cross_val  ?

- `cross_val_score()` already performs the training and validation splitting internally, so a separate `train_test_split()` is not mandatory before using cross-validation.

 
> #### Important Intuition
**train_test_split()** is generally used for a simple single evaluation.  
**cross_val_score()** is a more advanced and reliable evaluation technique because every data point gets a chance to become part of the test set

## Applying Feature Construction  

In [8]:
# instead of having seperate cols for Sipsp , Parch . We can have 1 col for both under the name Family 
x['Family_size'] = x['SibSp'] + x['Parch'] + 1    # 1 -> Counting Yourself

In [9]:
x.head()

,Pclass,Age,SibSp,Parch,Family_size
0,3,22.0,1,0,2
1,1,38.0,1,0,2
2,3,26.0,0,0,1
3,1,35.0,1,0,2
4,3,35.0,0,0,1


In [10]:
# Now lets Make Groups for better understing 
def myfunc(num):
    if num == 1:
        #alone
        return 0
    elif num >1 and num <=4:
        # small family
        return 1
    else:
        # large family
        return 2

In [11]:
# TO test Our function 
print(myfunc(1))
print(myfunc(4))
print(myfunc(6))

0
1
2


In [12]:
# It works well 
# Now lets apply this Func in df 
x['Family_type'] = x['Family_size'].apply(myfunc)

In [13]:
x.sample(5)

,Pclass,Age,SibSp,Parch,Family_size,Family_type
157,3,30.0,0,0,1,0
314,2,43.0,1,1,3,1
326,3,61.0,0,0,1,0
806,1,39.0,0,0,1,0
690,1,31.0,1,0,2,1


In [14]:
# Lets drop unnessaccry cols 
x.drop(columns=['SibSp','Parch','Family_size'],inplace=True)

In [15]:
x.head()

,Pclass,Age,Family_type
0,3,22.0,1
1,1,38.0,1
2,3,26.0,0
3,1,35.0,1
4,3,35.0,0


In [16]:
# Lets Check our score now 
np.mean(cross_val_score(LogisticRegression(),x,y,scoring='accuracy',cv=20))

np.float64(0.7003174603174602)

In [17]:
# it Imporves by 1% (69% -> 70%)

### Feature Splitting

In [18]:
df = pd.read_csv('train.csv')

In [19]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [37]:
df['Name']

0                                Braund, Mr. Owen Harris
1      Cumings, Mrs. John Bradley (Florence Briggs Th...
2                                 Heikkinen, Miss. Laina
3           Futrelle, Mrs. Jacques Heath (Lily May Peel)
4                               Allen, Mr. William Henry
                             ...                        
886                                Montvila, Rev. Juozas
887                         Graham, Miss. Margaret Edith
888             Johnston, Miss. Catherine Helen "Carrie"
889                                Behr, Mr. Karl Howell
890                                  Dooley, Mr. Patrick
Name: Name, Length: 891, dtype: object

In [38]:
# Will Split the Mr/Mrs text from this column , and Will make a seperate col for first & last name

In [43]:
# to Split 
df['Title'] = df['Name'].str.split(', ', expand=True)[1].str.split('.', expand=True)[0]
df['Title']

0        Mr
1       Mrs
2      Miss
3       Mrs
4        Mr
       ... 
886     Rev
887    Miss
888    Miss
889      Mr
890      Mr
Name: Title, Length: 891, dtype: object

#### Explaination of above code

In [40]:
# First Will Seperate by using delimeter ' , '
df['Name'].str.split(', ', expand=True)[1]   #-> Gets the Last name [0] | First name [1]

0                                 Mr. Owen Harris
1      Mrs. John Bradley (Florence Briggs Thayer)
2                                     Miss. Laina
3              Mrs. Jacques Heath (Lily May Peel)
4                               Mr. William Henry
                          ...                    
886                                   Rev. Juozas
887                          Miss. Margaret Edith
888                Miss. Catherine Helen "Carrie"
889                               Mr. Karl Howell
890                                   Mr. Patrick
Name: 1, Length: 891, dtype: object

In [41]:
# Then Will again seperate by using delimeter ' . '
df['Name'].str.split(', ', expand=True)[1].str.split('.',expand=True) 

# Defalut -> array output 
# Using expand = True -> Makes in Dataframe

,0,1,2
0,Mr,Owen Harris,None
1,Mrs,John Bradley (Florence Briggs Thayer),None
2,Miss,Laina,None
3,Mrs,Jacques Heath (Lily May Peel),None
4,Mr,William Henry,None
...,...,...,...
886,Rev,Juozas,None
887,Miss,Margaret Edith,None
888,Miss,"Catherine Helen ""Carrie""",None
889,Mr,Karl Howell,None


In [44]:
# now lets see together , Title & name 
df[['Title','Name']]

,Title,Name
0,Mr,"Braund, Mr. Owen Harris"
1,Mrs,"Cumings, Mrs. John Bradley (Florence Briggs Th..."
2,Miss,"Heikkinen, Miss. Laina"
3,Mrs,"Futrelle, Mrs. Jacques Heath (Lily May Peel)"
4,Mr,"Allen, Mr. William Henry"
...,...,...
886,Rev,"Montvila, Rev. Juozas"
887,Miss,"Graham, Miss. Margaret Edith"
888,Miss,"Johnston, Miss. Catherine Helen ""Carrie"""
889,Mr,"Behr, Mr. Karl Howell"


In [50]:
df.groupby('Title')['Survived'].mean().sort_values(ascending=False)

Title
the Countess    1.000000
Mlle            1.000000
Sir             1.000000
Ms              1.000000
Lady            1.000000
Mme             1.000000
Mrs             0.792000
Miss            0.697802
Master          0.575000
Col             0.500000
Major           0.500000
Dr              0.428571
Mr              0.156673
Jonkheer        0.000000
Rev             0.000000
Don             0.000000
Capt            0.000000
Name: Survived, dtype: float64

In [51]:
# We can see which Title has more survivale rate 

In [52]:
# Lets Create a new col for Married (0 -> not married , 1 -> Married )

df['Is_Married'] = 0     # Creates a col , where all values are 0 
df['Is_Married'].loc[df['Title'] == 'Mrs'] = 1   # Replace the 0 -> 1 , if title has Mrs in it 

C:\Users\Shubham\AppData\Local\Temp\ipykernel_29092\2819279589.py:4: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df['Is_Married'].loc[df['Title'] == 'Mrs'] = 1   # Replace the 0 -> 1 , if title has Mrs in it
C:\Users\Shubham\AppData\Local\

In [54]:
df['Is_Married']

0      0
1      1
2      0
3      1
4      0
      ..
886    0
887    0
888    0
889    0
890    0
Name: Is_Married, Length: 891, dtype: int64